In [0]:
import requests
import json
from datetime import datetime
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType, LongType

# Run store_key notebook to ensure secrets are configured
dbutils.notebook.run(
    "/Users/aadityrathore0344@gmail.com/financial-market-intelligence/Set_UP/store_key",
    timeout_seconds=60
)

# Load exchangerate API key from secrets (for reference, though we'll use CoinGecko)
exchange_key = dbutils.secrets.get(
    scope="fin-project",
    key="exchangerate_key"
)

print(f"✅ API key loaded: {exchange_key[:4]}****")

# Use CoinGecko API for cryptocurrency data (free tier, no API key needed)
coingecko_url = "https://api.coingecko.com/api/v3/coins/markets"

params = {
    "vs_currency": "usd",
    "order": "market_cap_desc",
    "per_page": 10,
    "page": 1,
    "sparkline": False,
    "price_change_percentage": "24h"
}

print(f"🔄 Fetching top 10 cryptocurrencies from CoinGecko API...")
response = requests.get(coingecko_url, params=params)

if response.status_code != 200:
    raise Exception(f"❌ API request failed: {response.status_code} - {response.text}")

data = response.json()

if not isinstance(data, list) or len(data) == 0:
    raise Exception(f"❌ No data returned from API")

# Extract cryptocurrency data
timestamp = datetime.now()
crypto_data = []

for i, crypto in enumerate(data, 1):
    crypto_data.append(Row(
        rank=i,
        symbol=crypto.get("symbol", "").upper(),
        name=crypto.get("name", ""),
        price_usd=crypto.get("current_price", 0.0),
        market_cap=crypto.get("market_cap", 0),
        volume_24h=crypto.get("total_volume", 0),
        price_change_24h_pct=crypto.get("price_change_percentage_24h", 0.0),
        fetch_timestamp=timestamp,
        source="coingecko"
    ))
    print(f"  {i}. {crypto.get('name')} ({crypto.get('symbol', '').upper()}): ${crypto.get('current_price', 0):,.2f}")

# Create DataFrame
schema = StructType([
    StructField("rank", LongType(), True),
    StructField("symbol", StringType(), False),
    StructField("name", StringType(), True),
    StructField("price_usd", DoubleType(), True),
    StructField("market_cap", LongType(), True),
    StructField("volume_24h", LongType(), True),
    StructField("price_change_24h_pct", DoubleType(), True),
    StructField("fetch_timestamp", TimestampType(), False),
    StructField("source", StringType(), True)
])

df = spark.createDataFrame(crypto_data, schema=schema)

print(f"\n✅ Fetched {df.count()} cryptocurrencies")

# Create database and schema if not exists
spark.sql("CREATE DATABASE IF NOT EXISTS financial_market")
spark.sql("CREATE SCHEMA IF NOT EXISTS financial_market.bronze")

# Write to bronze table
table_name = "financial_market.bronze.crypto"

df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(table_name)

print(f"✅ Data stored in {table_name}")
print(f"✅ Run timestamp: {timestamp}")

# Display sample
display(df)

In [0]:
# verify Bronze crypto table
df_verify = spark.sql("""
    SELECT 
        symbol,
        name,
        price_usd,
        market_cap,
        volume_24h,
        price_change_24h_pct,
        fetch_timestamp
    FROM financial_market.bronze.crypto
    ORDER BY rank
""")

print("✅ Bronze table — financial_market.bronze.crypto")
print(f"✅ Total rows: {df_verify.count()}")
display(df_verify)